# Train the proposed CNN-LSTM-Attention model

This is the main software deliverable. It runs the full training pipeline for our proposed model on a Colab T4 GPU and renders the attention figures inline.

**Runtime.** Set the Colab runtime to GPU before running (Runtime -> Change runtime type -> T4 GPU). End-to-end the notebook takes ~60-80 minutes on a free T4: most of that is the 30 training epochs, the rest is preprocessing stage-in and plotting. See `notebooks/COLAB_GUIDE.md` for the data-upload workflow.

**What this notebook does** (in order): mounts Drive, clones the repo, installs requirements, copies the preprocessed `windows.npz` to local SSD, trains via the CLI trainer in `src/training/train.py`, parses the train log into a metrics table and loss curves, renders an attention heatmap on a validation positive, and copies the resulting `runs/cnn_lstm_attention/<timestamp>/` directory back to Drive.

It does **not** redefine model or training logic. Everything important lives in `src/`; this notebook is a thin orchestrator.

## 1. Mount Google Drive

Skipped automatically when running outside Colab (e.g. locally on a workstation with a GPU).

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("not on Colab, skipping drive mount")

## 2. Clone the repo

Replace `<user>` with the GitHub user/org that hosts the repository. Skips the clone if a working tree already exists in the working directory.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/<user>/ecg_project.git"  # TODO: replace <user>
REPO_DIR = "/content/ecg_project" if "google.colab" in sys.modules else os.getcwd()

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    print(f"repo already at {REPO_DIR}, skipping clone")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 3. Install requirements

Colab ships most of these but the wfdb / pywavelets / shap / captum bits are not preinstalled. `-q` keeps the output short.

In [ ]:
!pip install -q -r requirements.txt

## 4. Stage the preprocessed data on local SSD

Drive I/O is slow and rate-limited; reading the 24k-window `windows.npz` from `/content/drive/.../sddb/` per epoch costs several minutes. Copying it once to local SSD makes the dataloader fast enough that we're GPU-bound rather than I/O-bound.

Adjust `DRIVE_SDDB` to wherever you uploaded the SDDB processed folder. The folder must contain `windows.npz` and `split.json` (see the COLAB guide for how those get there).

In [ ]:
import shutil
from pathlib import Path

DRIVE_SDDB = Path("/content/drive/MyDrive/ecg_project/data/processed/sddb")
LOCAL_SDDB = Path(REPO_DIR) / "data" / "processed" / "sddb"

if "google.colab" in sys.modules:
    LOCAL_SDDB.mkdir(parents=True, exist_ok=True)
    for name in ("windows.npz", "split.json"):
        src = DRIVE_SDDB / name
        dst = LOCAL_SDDB / name
        if not dst.exists():
            shutil.copy(src, dst)
            print(f"copied {src} -> {dst} ({dst.stat().st_size / 1e6:.1f} MB)")
        else:
            print(f"{dst} already staged")
else:
    print(f"not on Colab; assuming data already at {LOCAL_SDDB}")

## 5. Verify GPU

Sanity check before kicking off a 60+ minute run. `nvidia-smi` should show a T4 (or A100 if Colab Pro), and `torch.cuda.is_available()` should be True.

In [ ]:
!nvidia-smi -L
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 6. Train the proposed model

Invokes the CLI trainer rather than re-implementing the loop here, so this notebook and `scripts/03_train_all.sh` run the exact same code path. `--device cuda` forces GPU; if cuda is missing we want to fail loudly rather than silently fall back to CPU and run for 4 hours.

In [ ]:
!python -m src.training.train --model cnn_lstm_attention --device cuda --seed 42

## 7. Per-epoch metrics and loss curves

Pull the latest `runs/cnn_lstm_attention/<timestamp>/train.log`, parse the epoch lines into a table, and plot train vs. val loss alongside AUROC / AUPRC.

In [ ]:
import re
import glob
import pandas as pd
import matplotlib.pyplot as plt

run_dirs = sorted(glob.glob("runs/cnn_lstm_attention/*/"))
assert run_dirs, "no run directory found; did training complete?"
run_dir = Path(run_dirs[-1])
print("run_dir:", run_dir)

log_path = run_dir / "train.log"
pattern = re.compile(
    r"epoch (\d+): train_loss=([\d.]+) val_loss=([\d.]+) "
    r"val_auroc=([\d.]+) val_auprc=([\d.]+) lr=([\d.eE+-]+)"
)
rows = []
with log_path.open() as f:
    for line in f:
        m = pattern.search(line)
        if m:
            rows.append({
                "epoch": int(m.group(1)),
                "train_loss": float(m.group(2)),
                "val_loss": float(m.group(3)),
                "val_auroc": float(m.group(4)),
                "val_auprc": float(m.group(5)),
                "lr": float(m.group(6)),
            })
metrics = pd.DataFrame(rows)
metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(metrics["epoch"], metrics["train_loss"], label="train")
axes[0].plot(metrics["epoch"], metrics["val_loss"], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("focal loss"); axes[0].legend()
axes[0].set_title("loss curves")

axes[1].plot(metrics["epoch"], metrics["val_auroc"], label="val AUROC")
axes[1].plot(metrics["epoch"], metrics["val_auprc"], label="val AUPRC")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("score"); axes[1].legend()
axes[1].set_title("validation metrics")
fig.tight_layout()
plt.show()

## 8. Attention heatmap on a validation positive

Load `best.pt`, run the model on one validation positive, and plot the attention weights aligned to the 60-second ECG input. The CNN-LSTM-Attention model exposes its post-softmax attention as `model.last_attn_weights` after each forward pass; we upsample them back to sample resolution so the heatmap aligns with the raw signal.

If the val split contains no positives in your patient draw, the cell will print a warning and skip.

In [ ]:
import numpy as np
import torch

from src.data.dataset import make_dataloaders
from src.models.cnn_lstm_attention import CNNLSTMAttention

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(run_dir / "best.pt", map_location=device)
model = CNNLSTMAttention().to(device).eval()
model.load_state_dict(ckpt["model_state_dict"])

_, val_loader, _ = make_dataloaders(
    processed_root=Path("data/processed"),
    split_json=Path("data/processed/sddb/split.json"),
    batch_size=64,
    num_workers=0,
)
val_labels = val_loader.dataset.labels
pos_rows = np.where(val_labels > 0)[0]
if len(pos_rows) == 0:
    print("no positives in val split — skipping attention figure")
else:
    idx = int(pos_rows[0])
    window, label = val_loader.dataset[idx]
    with torch.no_grad():
        logit = model(window.unsqueeze(0).to(device))
    attn = model.last_attn_weights[0].cpu().numpy()      # (T',)
    signal = window.numpy()                              # (2, 15000)
    upsampled = np.repeat(attn, signal.shape[1] // attn.shape[0] + 1)[: signal.shape[1]]
    print(f"row={idx} label={label.item():.0f} logit={logit.item():+.3f}")

    fig, ax = plt.subplots(figsize=(11, 3))
    t = np.arange(signal.shape[1]) / 250.0
    ax.plot(t, signal[0], color="k", lw=0.6, label="lead I")
    ax2 = ax.twinx()
    ax2.fill_between(t, 0, upsampled, color="crimson", alpha=0.3, label="attention")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("ECG amplitude")
    ax2.set_ylabel("attention weight")
    ax.set_title(f"attention over validation positive (row {idx})")
    fig.tight_layout()
    plt.show()

## 9. Save the run back to Drive

Colab VMs get recycled, so the run directory has to land back on Drive to survive past disconnect. Skipped automatically when not on Colab.

In [ ]:
if "google.colab" in sys.modules:
    drive_runs = Path("/content/drive/MyDrive/ecg_project/runs/cnn_lstm_attention")
    drive_runs.mkdir(parents=True, exist_ok=True)
    dest = drive_runs / run_dir.name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(run_dir, dest)
    print(f"copied {run_dir} -> {dest}")
else:
    print("not on Colab; run dir stays at", run_dir)